# Milestone 4 — M4E: direct + diffuse integration

End-to-end M3 + M4 prototype:

1. Point light → refracted in-object segments (subset) → `S_clean`
2. FEM diffusion → `Φ`
3. Camera hit points → `I_diffuse`
4. Full source-ray budget → M3 direct channel → `I_direct`
5. `I_total = alpha * I_direct + I_diffuse`

Plan: [`plans/milestone_04/04_diffusion_plan.md`](../../plans/milestone_04/04_diffusion_plan.md).

## Scene notes

- Place the **light opposite the camera** so exit-face energy is visible in `I_direct`.
- **Ray budget:** dense rays for direct; random subset (scaled weights) for expensive deposition/solve.
- **`alpha` / `exitance_scale`** are explicit compositing knobs — recorded in metadata.
- **No `cache_dir`** on mesh generation here (live Netgen handle required).


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

!pip install --quiet --no-cache-dir "{ROOT}[fem]" -c "{ROOT}/requirements.txt"

from gummybear.paths import display_path

print(f"ROOT={{display_path(ROOT)}}")



In [2]:
import numpy as np

from gummybear.geometry import inspect_stl
from gummybear.optics import (
    FORWARD_MODEL_TIER,
    OpticalMaterialConfig,
    PointLightConfig,
    SourceSamplingParams,
    compose_hybrid_image,
    compute_refractive_direct_image,
    deposit_ray_source,
    generate_diffusion_mesh,
    in_object_segments_from_rays,
    make_source_ray_bundle,
    refract_ray_bundle,
    sample_diffuse_image,
    solve_diffusion,
)
from gummybear.rays import PinholeCameraConfig, first_visible_hits_with_points, make_camera_rays
from gummybear_validation.helpers import (
    assert_live_netgen_mesh,
    assert_m4e_hybrid_checks,
    build_diffusion_ray_subset,
    m4e_metadata_template,
    print_diffusion_mesh_summary,
    print_m4e_metadata,
    write_m4e_artifacts,
)
from gummybear_validation.plotting import plot_hybrid_panels


In [ ]:
RUN_ID = "m4e_end_to_end_direct_plus_diffuse"
STL_PATH = ROOT / "cad" / "proto_bear.stl"
OUT_DIR = ROOT / "data" / "generated" / "m4e" / RUN_ID
TARGET_ELEMENTS = 2000

material = OpticalMaterialConfig(n_refractive=1.33, mu_scatter=0.2, mu_absorption=0.0001, g=0.0)
extrapolation_length = 5.0
alpha = 0.0
exitance_scale = 1.0
direct_scale = 1.0
n_environment = 1.0

camera = PinholeCameraConfig(
    camera_position=(0.0, -50.0, 0.0),
    look_at=(0.0, 0.0, 0.0),
    up=(0.0, 0.0, 1.0),
    fov_deg=35.0,
    resolution=128,
)
# Opposite side of camera for transmission silhouette in I_direct.
light = PointLightConfig(position=(5.0, 45.0, 15.0), intensity=200.0)
N_DIRECT_RAYS = 50_000
N_DIFFUSE_RAYS = 500
RAY_SEED = 0

print("forward_model_tier:", FORWARD_MODEL_TIER)
print("alpha / exitance / direct_scale:", alpha, exitance_scale, direct_scale)


In [ ]:
inspection = inspect_stl(STL_PATH)
mesh = inspection["mesh"]
diff_mesh = generate_diffusion_mesh(mesh, target_elements=TARGET_ELEMENTS)
print_diffusion_mesh_summary(diff_mesh)
assert_live_netgen_mesh(diff_mesh)


In [ ]:
light_sampling = SourceSamplingParams(mode="point_uniform", n_rays=N_DIRECT_RAYS, seed=RAY_SEED)
source_rays = make_source_ray_bundle(light, mesh_bbox=mesh.bounds, sampling=light_sampling)
diffusion_rays, weight_scale = build_diffusion_ray_subset(source_rays, N_DIFFUSE_RAYS, seed=RAY_SEED)

cam_rays = make_camera_rays(camera)
H, W = cam_rays.sample_shape
cam_valid, _depth, cam_faces, cam_points = first_visible_hits_with_points(mesh, cam_rays)
camera_mask = cam_valid.reshape(H, W)
hit_faces_img = cam_faces.reshape(H, W)
view_dirs = -cam_rays.directions.reshape(H, W, 3)

print("source rays:", source_rays.n_rays, "diffusion subset:", diffusion_rays.n_rays)
print("camera hit fraction:", float(cam_valid.mean()))


In [ ]:
entry = refract_ray_bundle(mesh, diffusion_rays, n_from=n_environment, n_to=material.n_refractive)
segments = in_object_segments_from_rays(mesh, entry.rays)
print("refracted / segments:", entry.n_refracted, segments.n_rays)

deposit = deposit_ray_source(diff_mesh, segments, material=material)
energy_check = float(np.sum(deposit.S_clean * diff_mesh.volumes))
print("sum(S_clean * vol) ~ total_scattered:", energy_check, deposit.total_scattered)

solved = solve_diffusion(
    diff_mesh,
    S_clean=deposit.S_clean,
    D=material.diffusion_coefficient,
    mu_a=material.mu_a,
    extrapolation_length=extrapolation_length,
)
print("Phi residual_norm:", solved.residual_norm)


In [7]:
diffuse = sample_diffuse_image(
    diff_mesh,
    solved.Phi_nodes,
    hit_points=cam_points,
    valid_mask=cam_valid,
    sample_shape=(H, W),
    exitance_scale=exitance_scale,
    interpolate=False,
)
I_diffuse = diffuse.I_diffuse

direct = compute_refractive_direct_image(
    mesh,
    source_rays,
    material,
    hit_faces_img,
    view_dirs,
    direct_scale=direct_scale,
    apply_attenuation=True,
    camera_mask=camera_mask,
)
I_direct = direct.I_direct


In [ ]:
hybrid = compose_hybrid_image(I_direct, I_diffuse, alpha=alpha, camera_mask=camera_mask)

plot_hybrid_panels(
    hybrid,
    camera_mask,
    forward_model_tier=FORWARD_MODEL_TIER,
    show_alpha_sweep=True,
)

assert_m4e_hybrid_checks(hybrid, I_diffuse, camera_mask)


In [ ]:
meta = m4e_metadata_template(
    run_id=RUN_ID,
    forward_model=FORWARD_MODEL_TIER,
    alpha=alpha,
    exitance_scale=exitance_scale,
    direct_scale=direct_scale,
    stl_path=STL_PATH,
    target_elements=TARGET_ELEMENTS,
    diff_mesh=diff_mesh,
    material=material,
    extrapolation_length=extrapolation_length,
    robin_boundary_model=solved.robin_boundary_model,
    light=light,
    source_rays=source_rays,
    diffusion_rays=diffusion_rays,
    weight_scale=weight_scale,
    ray_seed=RAY_SEED,
    camera=camera,
    sample_shape=(H, W),
    hit_fraction=float(cam_valid.mean()),
    deposit=deposit,
    n_segments=segments.n_rays,
    hybrid=hybrid,
)
write_m4e_artifacts(OUT_DIR, hybrid=hybrid, deposit=deposit, solved=solved, metadata=meta, camera_mask=camera_mask)
print_m4e_metadata(meta)
